# 03 — Evaluation: LPIPS + Condition Accuracy
Computes both required metrics for all models and prints a comparison table.
Run this after `02_inference.ipynb` has populated the `outputs/` folders.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q lpips torch torchvision pillow pandas
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
if IN_COLAB:
    BASE = '/content/drive/My Drive/CIS_5190_group_project'
else:
    BASE = '..'

EVAL_CSV        = f'{BASE}/lpips_eval_set.csv'
ALIGNED_DIR     = f'{BASE}/aligned'
CLASSIFIER_CKPT = f'{BASE}/checkpoints/classifier_best.pt'

# Maps model name → (output folder, filename suffix)
# Comment out any model whose outputs don't exist yet
MODELS = {
    'SD Baseline':       (f'{BASE}/outputs/sd_baseline',     '_sd_baseline.jpg'),
    'InstructPix2Pix':   (f'{BASE}/outputs/instructpix2pix', '_ip2p.jpg'),
    'ControlNet':        (f'{BASE}/outputs/controlnet',      '_controlnet.jpg'),
    'ControlNet + LoRA': (f'{BASE}/outputs/controlnet_lora', '_controlnet_lora.jpg'),
}

In [ ]:
import os
import torch
import lpips
import pandas as pd
import numpy as np
from PIL import Image
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

loss_fn = lpips.LPIPS(net='alex').to(device)
eval_df = pd.read_csv(EVAL_CSV)

lpips_tf = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

def compute_lpips(gt_path, gen_path):
    gt  = lpips_tf(Image.open(gt_path).convert('RGB')).unsqueeze(0).to(device)
    gen = lpips_tf(Image.open(gen_path).convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        return loss_fn(gt, gen).item()

print('LPIPS ready.')

In [ ]:
import sys
if IN_COLAB:
    import os
    if not os.path.exists('/content/image-style-transfer'):
        !git clone https://github.com/YOUR_ORG/image-style-transfer /content/image-style-transfer
    sys.path.insert(0, '/content/image-style-transfer')
else:
    sys.path.insert(0, os.path.join(BASE, 'image-style-transfer'))

from scripts.train_classifier import DualHeadResNet, TOD_CLASSES, WX_CLASSES

clf_ckpt = torch.load(CLASSIFIER_CKPT, map_location=device)
classifier = DualHeadResNet().to(device)
classifier.load_state_dict(clf_ckpt['model_state_dict'])
classifier.eval()

clf_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def classify(img_path):
    img = clf_tf(Image.open(img_path).convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        tod_logits, wx_logits = classifier(img)
    return TOD_CLASSES[tod_logits.argmax().item()], WX_CLASSES[wx_logits.argmax().item()]

print('Classifier loaded.')

In [ ]:
summary = []

for model_name, (out_dir, suffix) in MODELS.items():
    lpips_scores = []
    correct_tod = correct_wx = total = 0

    for _, row in eval_df.iterrows():
        stem     = os.path.splitext(row['target_file'])[0]
        gen_path = os.path.join(out_dir, stem + suffix)
        gt_path  = os.path.join(ALIGNED_DIR, row['warped_path'])

        if not os.path.exists(gen_path):
            continue

        lpips_scores.append(compute_lpips(gt_path, gen_path))
        pred_tod, pred_wx = classify(gen_path)
        correct_tod += int(pred_tod == row['target_tod'].lower())
        correct_wx  += int(pred_wx  == row['target_weather'].lower())
        total += 1

    if total == 0:
        print(f'[SKIP] {model_name} — no output files found')
        continue

    summary.append({
        'Model':           model_name,
        'LPIPS ↓':         round(np.mean(lpips_scores), 4),
        'LPIPS std':       round(np.std(lpips_scores), 4),
        'ToD Acc ↑':       round(correct_tod / total, 3),
        'Weather Acc ↑':   round(correct_wx  / total, 3),
        'N':               total,
    })

results_df = pd.DataFrame(summary)
print(results_df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

n = len(results_df)
x = np.arange(n)
labels = results_df['Model']

fig, axes = plt.subplots(1, 2, figsize=(5 * n, 5))

axes[0].bar(x, results_df['LPIPS ↓'], yerr=results_df['LPIPS std'], capsize=5)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels, rotation=15, ha='right')
axes[0].set_title('LPIPS (lower is better)'); axes[0].set_ylabel('LPIPS')

axes[1].bar(x - 0.2, results_df['ToD Acc ↑'],     width=0.4, label='Time of Day')
axes[1].bar(x + 0.2, results_df['Weather Acc ↑'], width=0.4, label='Weather')
axes[1].set_xticks(x); axes[1].set_xticklabels(labels, rotation=15, ha='right')
axes[1].set_title('Condition Accuracy (higher is better)')
axes[1].set_ylabel('Accuracy'); axes[1].legend()

plt.tight_layout()
os.makedirs(f'{BASE}/outputs', exist_ok=True)
plt.savefig(f'{BASE}/outputs/evaluation_summary.png', dpi=150, bbox_inches='tight')
plt.show()